# CT Noise Synthesis Pipeline

This notebook extracts real noise-residual texture from a set of CT images and uses it,
perturbed with additive Gaussian + Poisson noise, to synthesize noisy versions of clean images.


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from tqdm import tqdm

## 1. Configuration

Edit the paths below to match your environment.

In [ ]:
# ---- Paths ----
INPUT_FOLDER  = os.path.expanduser("~/Desktop/INPUT_FOLDER")     # raw source images
PREPRO_FOLDER = os.path.expanduser("~/Desktop/PREPRO_FOLDER")       # preprocessed (resized/normalized) images
NOISE_FOLDER  = os.path.expanduser("~/Desktop/noise")       # extracted noise-residual templates
OUTPUT_FOLDER = os.path.expanduser("~/Desktop/synthetic_noisy")  # final synthetic noisy images

IMG_SIZE = (256, 256)
VALID_EXT = (".bmp", ".jpg", ".jpeg", ".png", '.tif', '.tiff')

os.makedirs(PREPRO_FOLDER, exist_ok=True)
os.makedirs(NOISE_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)


## 2. Preprocessing

Load each raw image, convert to grayscale, resize, and normalize to `[0, 1]`.

In [ ]:
def preprocess_image(image_path, img_size=IMG_SIZE):
    """Load an image as grayscale, resize, and scale to [0, 1]."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")
    img_resized = cv2.resize(img, img_size)
    return img_resized / 255.0


def run_preprocessing(input_folder, output_folder, img_size=IMG_SIZE):
    filenames = [f for f in os.listdir(input_folder) if f.lower().endswith(VALID_EXT)]

    for filename in tqdm(filenames, desc="Preprocessing"):
        image_path = os.path.join(input_folder, filename)
        preprocessed_image = preprocess_image(image_path, img_size)
        output_path = os.path.join(output_folder, filename)
        cv2.imwrite(output_path, (preprocessed_image * 255).astype(np.uint8))

    print(f"Preprocessing completed: {len(filenames)} images written to {output_folder}")


run_preprocessing(INPUT_FOLDER, PREPRO_FOLDER)


## 3. Noise Analysis

Extract a high-frequency residual map from each preprocessed image by subtracting a Gaussian-blurred
version of itself. This residual is a mix of real sensor noise, edges, and fine anatomical detail -
it's a proxy "noise template", not pure noise, but it's the most genuine, data-driven noise source
available here, and it's what actually feeds the synthesis step below.

In [ ]:
def extract_noise_residual(image_path, sigma=1):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    denoised_img = gaussian_filter(img, sigma=sigma)
    residual = img - denoised_img
    residual_normalized = cv2.normalize(residual, None, 0, 255, cv2.NORM_MINMAX)
    return img, denoised_img, residual_normalized


def run_noise_analysis(input_folder, output_folder, sigma=1):
    filenames = [f for f in os.listdir(input_folder) if f.lower().endswith(VALID_EXT)]

    for filename in tqdm(filenames, desc="Extracting noise residuals"):
        image_path = os.path.join(input_folder, filename)
        _, _, residual_normalized = extract_noise_residual(image_path, sigma)
        output_path = os.path.join(output_folder, f"noise_{filename}")
        cv2.imwrite(output_path, residual_normalized.astype(np.uint8))

    print(f"Noise analysis completed: residual maps written to {output_folder}")


run_noise_analysis(PREPRO_FOLDER, NOISE_FOLDER)


## 4. Load clean images and noise templates

In [ ]:
def load_image_folder(folder, img_size=IMG_SIZE):
    images = []
    for fname in sorted(os.listdir(folder)):
        if fname.lower().endswith(VALID_EXT):
            img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, img_size)
            images.append(img.astype(np.float32) / 255.0)
    return np.array(images)


clean_images    = load_image_folder(PREPRO_FOLDER)   # clean preprocessed CT images
noise_templates = load_image_folder(NOISE_FOLDER)    # real residual/noise textures

print(f"Loaded {len(clean_images)} clean images and {len(noise_templates)} noise templates.")


## 5. Synthetic noise generation

Perturb the real residual templates with additive Gaussian + Poisson noise to build a pool of
varied synthetic noise patterns. 

Parameters follow the pipeline figure: σ_g = `gaussian_scale` = 0.005, λ = `poisson_lambda` = 0.1.


In [ ]:
def generate_synthetic_noise(noise_templates, gaussian_scale=0.005, poisson_lambda=0.1, rng=rng):
    templates = noise_templates.astype(np.float64)

    gaussian_noise = rng.normal(loc=0.0, scale=gaussian_scale, size=templates.shape)
    poisson_noise = rng.poisson(lam=poisson_lambda, size=templates.shape) / 255.0

    synthetic_noise = templates + gaussian_noise + poisson_noise
    return np.clip(synthetic_noise, 0, 1)


synthetic_noise_pool = generate_synthetic_noise(noise_templates, gaussian_scale=0.005, poisson_lambda=0.1)
print("Synthetic noise pool shape:", synthetic_noise_pool.shape)

# Sanity-check visualization: original template vs. perturbed version
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(noise_templates[0], cmap='gray')
plt.title("Original noise template")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(synthetic_noise_pool[0], cmap='gray')
plt.title("Perturbed synthetic noise")
plt.axis('off')
plt.tight_layout()
plt.show()


## 6. Apply synthetic noise to clean images

For each clean image $x_i$, randomly sample a synthetic noise pattern $S_k$ ($k \sim \mathcal{U}\{1,\dots,N\}$), apply a contrast gain
`alpha` and add the noise at `noise_factor` strength:

$$\tilde{y}_i = \mathrm{clip}_{[0,1]}(\alpha x_i + \beta S_k), \quad \alpha = 0.63,\ \beta = 0.325$$

then apply a Poisson pass: $y_i = \mathrm{clip}\big(\tfrac{1}{255}\mathrm{Pois}(255\,\tilde{y}_i)\big)$.

In [ ]:
def apply_noise_to_images(clean_images, noise_pool, alpha=0.63, noise_factor=0.325, apply_poisson=True, rng=rng):
    num_images = clean_images.shape[0]
    indices = rng.integers(0, noise_pool.shape[0], size=num_images)
    selected_noise = noise_pool[indices]

    noisy_images = alpha * clean_images + noise_factor * selected_noise
    noisy_images = np.clip(noisy_images, 0, 1)

    if apply_poisson:
        noisy_images = rng.poisson(noisy_images * 255) / 255.0
        noisy_images = np.clip(noisy_images, 0, 1)

    return noisy_images


noisy_images = apply_noise_to_images(clean_images, synthetic_noise_pool, alpha=0.63, noise_factor=0.325)
print(f"Generated {noisy_images.shape[0]} synthetic noisy images.")


## 7. Save outputs

In [ ]:
def save_images(images, filenames, output_folder):
    for img, fname in zip(images, filenames):
        out_path = os.path.join(output_folder, fname)
        cv2.imwrite(out_path, (img * 255).astype(np.uint8))


source_filenames = sorted(f for f in os.listdir(PREPRO_FOLDER) if f.lower().endswith(VALID_EXT))
save_images(noisy_images, source_filenames, OUTPUT_FOLDER)
print(f"Saved {len(noisy_images)} noisy images to {OUTPUT_FOLDER}")


## 8. Visual comparison

In [ ]:
n_show = min(4, len(clean_images))
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
if n_show == 1:
    axes = axes.reshape(2, 1)

for i in range(n_show):
    axes[0, i].imshow(clean_images[i], cmap='gray')
    axes[0, i].set_title("Clean")
    axes[0, i].axis('off')

    axes[1, i].imshow(noisy_images[i], cmap='gray')
    axes[1, i].set_title("Synthetic Noisy")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()
